In [ ]:
KE_SRC = 'import os, glob, time, gc\nos.environ.setdefault(\'HF_HUB_OFFLINE\', \'1\')\nos.environ.setdefault(\'TRANSFORMERS_OFFLINE\', \'1\')\nos.environ.setdefault(\'HF_HUB_DISABLE_TELEMETRY\', \'1\')\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\nimport timm\ntorch.backends.cudnn.benchmark = True\ntorch.backends.cuda.matmul.allow_tf32 = True\nIMG = 336\nCROP_MM = 140.0\nSLOTS = [(\'Sagittal\', 1, 18), (\'Sagittal\', 0, 14), (\'Coronal\', 1, 12), (\'Coronal\', 0, 8), (\'Axial\', -1, 12)]\nMAXS = sum((s[2] for s in SLOTS))\nK_EVAL = 42\nNORM = \'imagenet\'\nLAB = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\', \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\n_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)\n_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)\nARMS = [{\'file\': \'raptor_ft_coatnet_v5_full_swa.pt\', \'arch\': \'coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k\', \'res\': 384, \'w\': 1.0}]\n\ndef build_backbone(arch, pretrained=False):\n    hybrid = arch.startswith((\'maxvit\', \'maxxvit\', \'coatnet\', \'coat_\', \'convnext\'))\n    is_vit = not hybrid and any((k in arch for k in (\'vit\', \'deit\', \'dinov2\', \'eva\', \'beit\')))\n    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)\n    if is_vit:\n        kw.update(global_pool=\'token\', dynamic_img_size=True)\n    else:\n        kw.update(global_pool=\'avg\')\n    return timm.create_model(arch, **kw)\n\nclass RaptorClassifier(nn.Module):\n\n    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x):\n        B, K = x.shape[:2]\n        f = self.backbone(x.flatten(0, 1))\n        return f.view(B, K, -1)\n\n    def head(self, feats):\n        h = self.norm(feats)\n        a = self.att(h)\n        a = torch.softmax(a, dim=1)\n        pooled = torch.einsum(\'bkn,bkf->bnf\', a, h)\n        logits = (pooled * self.clsW).sum(-1) + self.clsb\n        return logits\n\n    def forward(self, x):\n        return self.head(self.encode(x))\n\ndef load_model(pt_path, arch_default, res_default, device, ngpu=1):\n    ck = torch.load(pt_path, map_location=\'cpu\', weights_only=False)\n    arch = ck.get(\'arch\', arch_default)\n    ck_res = int(ck.get(\'res\', res_default))\n    bb = build_backbone(arch, pretrained=False)\n    model = RaptorClassifier(bb, F_dim=bb.num_features)\n    model.load_state_dict(ck[\'model\'], strict=True)\n    model.eval().to(device)\n    del ck\n    gc.collect()\n    return (model, ck_res)\n\ndef _eval_centers(mask, D, k):\n    valid = np.where(mask > 0)[0]\n    if len(valid) < 3:\n        valid = np.arange(min(3, D))\n    lo, hi = (int(valid.min()), int(valid.max()))\n    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]\n    if not cs:\n        cs = [max(1, min((lo + hi) // 2, D - 2))]\n    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)\n    return [cs[i] for i in idx]\n\ndef eval_windows(vol, mask, k, res, norm=NORM):\n    D = vol.shape[0]\n    cs = _eval_centers(mask, D, k)\n    wins = np.empty((len(cs), 3, res, res), np.float32)\n    for j, c in enumerate(cs):\n        c = max(1, min(c, D - 2))\n        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0\n        t = torch.from_numpy(tri)\n        if t.shape[-1] != res:\n            t = F.interpolate(t[None], size=(res, res), mode=\'bilinear\', align_corners=False)[0]\n        wins[j] = t.numpy()\n    x = torch.from_numpy(wins)\n    if norm == \'imagenet\':\n        x = (x - _MEAN) / _STD\n    return x\n\n@torch.no_grad()\ndef infer_probs(model, xwins, device):\n    x = xwins.unsqueeze(0).to(device)\n    use_cuda = device != \'cpu\' and str(device).startswith(\'cuda\')\n    if use_cuda:\n        try:\n            with torch.autocast(\'cuda\', dtype=torch.float16):\n                o = torch.sigmoid(model(x).float())\n            return o[0].cpu().numpy()\n        except RuntimeError:\n            torch.cuda.empty_cache()\n            o = torch.sigmoid(model(x).float())\n            return o[0].cpu().numpy()\n    o = torch.sigmoid(model(x).float())\n    return o[0].cpu().numpy()\n\ndef rankpct(x):\n    order = x.argsort(0).argsort(0).astype(np.float64)\n    return order / max(1, x.shape[0] - 1)\n\ndef _make_reader():\n    import pydicom, cv2\n    from pydicom.pixel_data_handlers.util import apply_modality_lut\n\n    def order_and_meta(sdir):\n        fs = glob.glob(sdir + \'/*.dcm\')\n        recs = []\n        ps_list = []\n        for f in fs:\n            try:\n                h = pydicom.dcmread(f, stop_before_pixels=True)\n                iop = getattr(h, \'ImageOrientationPatient\', None)\n                ipp = getattr(h, \'ImagePositionPatient\', None)\n                if iop is not None and ipp is not None and (len(iop) == 6):\n                    r = np.array(iop[:3], float)\n                    c = np.array(iop[3:], float)\n                    n = np.cross(r, c)\n                    pos = float(np.dot(np.array(ipp, float), n))\n                else:\n                    pos = float(getattr(h, \'InstanceNumber\', 0) or 0)\n                ps = getattr(h, \'PixelSpacing\', None)\n                ps = float(ps[0]) if ps is not None else 0.5\n                ps_list.append(ps)\n                recs.append((pos, f, ps))\n            except Exception:\n                recs.append((0.0, f, 0.5))\n        recs.sort(key=lambda x: x[0])\n        med_ps = float(np.median(ps_list)) if ps_list else 0.5\n        return ([(f, ps) for _, f, ps in recs], med_ps)\n\n    def read_px(f):\n        d = pydicom.dcmread(f)\n        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n        if str(getattr(d, \'PhotometricInterpretation\', \'\')) == \'MONOCHROME1\':\n            a = a.max() - a\n        return a\n\n    def mm_crop_resize(a, ps):\n        h, w = a.shape\n        cpx = int(round(CROP_MM / max(ps, 0.001)))\n        cpx = min(cpx, min(h, w))\n        y0 = (h - cpx) // 2\n        x0 = (w - cpx) // 2\n        a = a[y0:y0 + cpx, x0:x0 + cpx]\n        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)\n    return (order_and_meta, read_px, mm_crop_resize)\n\ndef _pick_series_for_slot(rows, plane, fluid, used):\n    cands = [r for r in rows if r[\'Anatomical_Plane\'] == plane and r[\'SeriesInstanceUID\'] not in used]\n    if fluid in (0, 1):\n        pref = [r for r in cands if int(r.get(\'Fluid_Sensitive\', 0) or 0) == fluid]\n        if pref:\n            return pref[0]\n    return cands[0] if cands else None\n\ndef build_study(sid, ser_records, tsdir, reader):\n    order_and_meta, read_px, mm_crop_resize = reader\n    rows = ser_records.get(sid, [])\n    vol = np.zeros((MAXS, IMG, IMG), np.uint8)\n    idx = 0\n    used = set()\n    for plane, fluid, k in SLOTS:\n        r = _pick_series_for_slot(rows, plane, fluid, used)\n        if r is None:\n            idx += k\n            continue\n        used.add(r[\'SeriesInstanceUID\'])\n        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r[\'SeriesInstanceUID\']}")\n        if not files:\n            idx += k\n            continue\n        n = len(files)\n        lo, hi = (int(n * 0.06), int(n * 0.94) - 1)\n        hi = max(hi, lo)\n        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k\n        arrs = []\n        pss = []\n        for p in picks:\n            fp, ps = files[min(p, n - 1)]\n            try:\n                arrs.append(read_px(fp))\n                pss.append(ps)\n            except Exception:\n                arrs.append(None)\n                pss.append(med_ps)\n        valid = [a for a in arrs if a is not None]\n        if valid:\n            allpx = np.concatenate([a.ravel() for a in valid])\n            loq, hiq = np.percentile(allpx, [2.0, 98.0])\n        else:\n            loq, hiq = (0.0, 1.0)\n        for a, ps in zip(arrs, pss):\n            if idx >= MAXS:\n                break\n            if a is None:\n                idx += 1\n                continue\n            aw = np.clip((a - loq) / (hiq - loq + 1e-06), 0, 1)\n            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)\n            vol[idx] = (aw * 255).astype(np.uint8)\n            idx += 1\n        if idx >= MAXS:\n            break\n    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)\n    return (vol, mask)\n\ndef find_test_root():\n    cands = [\'/kaggle/input/competitions/rsna-knee-abnormality-detection\', \'/kaggle/input/rsna-knee-abnormality-detection\']\n    for b in cands:\n        if os.path.exists(b + \'/test.csv\'):\n            return b\n    for d, _, f in os.walk(\'/kaggle/input\'):\n        if \'test.csv\' in f and (os.path.isdir(d + \'/test_series\') or os.path.isdir(d + \'/test_images\')):\n            return d\n    for d, _, f in os.walk(\'/kaggle/input\'):\n        if \'test.csv\' in f:\n            return d\n    raise RuntimeError(\'no test root under /kaggle/input\')\n\ndef find_weight_file(fname):\n    direct = [f\'/kaggle/input/raptor-knee-arms/{fname}\', f\'/kaggle/input/raptor-knee-arms/1/{fname}\', f\'/kaggle/input/raptor-cnn336/{fname}\']\n    for p in direct:\n        if os.path.exists(p):\n            return p\n    for d in sorted(glob.glob(\'/kaggle/input/*/\')):\n        if \'competition\' in d.lower():\n            continue\n        hits = glob.glob(os.path.join(d, \'**\', fname), recursive=True)\n        if hits:\n            return hits[0]\n    raise RuntimeError(f\'{fname} not found under /kaggle/input\')\n\ndef main():\n    import pandas as pd\n    t0 = time.time()\n    dev = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    ngpu = torch.cuda.device_count()\n    print(f\'device {dev} | gpus {ngpu} | torch {torch.__version__}\', flush=True)\n    ROOT = find_test_root()\n    tsdir = ROOT + \'/test_series\'\n    if not os.path.isdir(tsdir):\n        tsdir = ROOT + \'/test_images\'\n    print(\'test root:\', ROOT, \'| series dir:\', tsdir, flush=True)\n    test = pd.read_csv(ROOT + \'/test.csv\')\n    test[\'StudyInstanceUID\'] = test[\'StudyInstanceUID\'].astype(str)\n    test_ids = test[\'StudyInstanceUID\'].tolist()\n    tser = pd.read_csv(ROOT + \'/test_series.csv\')\n    tser[\'StudyInstanceUID\'] = tser[\'StudyInstanceUID\'].astype(str)\n    tser[\'SeriesInstanceUID\'] = tser[\'SeriesInstanceUID\'].astype(str)\n    SER = {k: v.to_dict(\'records\') for k, v in tser.groupby(\'StudyInstanceUID\')}\n    print(f\'test studies {len(test_ids)} | test series {len(tser)}\', flush=True)\n    sub_cols = [\'StudyInstanceUID\'] + LAB\n    ssub = os.path.join(ROOT, \'sample_submission.csv\')\n    if os.path.exists(ssub):\n        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)\n    reader = _make_reader()\n    N = len(test_ids)\n    A = len(ARMS)\n    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]\n    for a, arm in enumerate(ARMS):\n        wp = find_weight_file(arm[\'file\'])\n        model, res = load_model(wp, arm[\'arch\'], arm[\'res\'], dev)\n        print(f"[arm {a}] loaded {arm[\'file\']} | res {res} | {time.time() - t0:.0f}s", flush=True)\n        for i, sid in enumerate(test_ids):\n            try:\n                vol, mask = build_study(sid, SER, tsdir, reader)\n                xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM)\n                arm_probs[a][i] = infer_probs(model, xw, dev)\n                del vol, mask, xw\n            except Exception as e:\n                print(f\'  [arm {a}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})\', flush=True)\n            if (i + 1) % 100 == 0 or i + 1 == N:\n                print(f\'  [arm {a}] {i + 1}/{N} | {time.time() - t0:.0f}s\', flush=True)\n        del model\n        gc.collect()\n        if str(dev).startswith(\'cuda\'):\n            torch.cuda.empty_cache()\n        print(f\'[arm {a}] done + freed | {time.time() - t0:.0f}s\', flush=True)\n    _w = np.array([float(a.get(\'w\', 1.0)) for a in ARMS], dtype=np.float64)\n    _w = _w / _w.sum()\n    print(f"[blend] weighted rank-mean w={dict(zip([a[\'file\'] for a in ARMS], _w.round(4)))}", flush=True)\n    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]), axes=(0, 0))\n    if not np.isfinite(ranks).all():\n        ranks[~np.isfinite(ranks)] = 0.5\n    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)\n    sub.insert(0, \'StudyInstanceUID\', test_ids)\n    sub = sub[sub_cols]\n    assert list(sub.columns) == sub_cols, \'column order drift\'\n    assert sub[\'StudyInstanceUID\'].tolist() == test_ids, \'row identity drift\'\n    assert np.isfinite(sub[LAB].values).all()\n    out = \'/kaggle/working/_raptor.csv\'\n    sub.to_csv(out, index=False)\n    print(\'wrote\', out, \'|\', len(sub), \'rows x\', len(sub.columns), \'cols\', flush=True)\n    print(sub.head().to_string(index=False), flush=True)\n    print(f\'DONE {time.time() - t0:.0f}s\', flush=True)'
WORKER_SRC = '\nimport os, sys, time, glob, math, random, hashlib, json, copy\nimport numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F\nt0 = time.time()\ndef tlog(m): print(f\'[t={time.time()-t0:.0f}s] {m}\', flush=True)\nCFG = json.loads(os.environ[\'CFG\']); DEVKIND = CFG[\'dev\']\nif DEVKIND == \'tpu\':\n    import torch_xla.core.xla_model as xm\n    DEV = xm.xla_device(); AC = lambda: torch.autocast(\'xla\', dtype=torch.bfloat16)\n    def mark(): xm.mark_step()\nelse:\n    DEV = \'cuda\'; AC = lambda: torch.autocast(\'cuda\', dtype=torch.float16)\n    def mark(): pass\nNW = int(os.environ.get(\'NW\', \'4\'))\nRES, KW, EPOCHS, SEED, NFOLD = int(CFG[\'res\']), int(CFG[\'k\']), int(CFG[\'epochs\']), int(CFG[\'seed\']), int(CFG[\'nfold\'])\nFOLDS = [int(x) for x in os.environ[\'FOLDS\'].split(\',\') if x != \'\']\nBS, ACC, K_EVAL = int(CFG[\'bs\']), int(CFG[\'acc\']), int(CFG[\'k_eval\'])\nLR_ENC, LR_HEAD, WD = 1e-4, 5e-4, 0.02\nBOUNDS = [(0, 18), (18, 32), (32, 44), (44, 52), (52, 64)]; NS = len(BOUNDS)\nKE = {\'__name__\': \'_ke\'}; exec(compile(open(\'/kaggle/working/_ke_src.py\').read(), \'<ke>\', \'exec\'), KE)\nLAB = KE[\'LAB\']; N_T = len(LAB)\ndef _bglob(name):\n    return [p for d in (\'/kaggle/input/*/\', \'/kaggle/input/*/*/\', \'/kaggle/input/*/*/*/\', \'/kaggle/input/*/*/*/*/\') for p in glob.glob(d + name)]\n\nROOT = KE[\'find_test_root\']()\ntrain = pd.read_csv(ROOT + \'/train.csv\'); train[\'StudyInstanceUID\'] = train[\'StudyInstanceUID\'].astype(str)\ntser = pd.read_csv(ROOT + \'/train_series.csv\'); tser[\'StudyInstanceUID\'] = tser[\'StudyInstanceUID\'].astype(str)\nHAS = set(tser[\'StudyInstanceUID\']); GOLD = set(train.loc[train[LAB].notna().all(axis=1), \'StudyInstanceUID\'])\nALL_IDS = [s for s in train[\'StudyInstanceUID\'] if s in HAS]\nMOUNTED = {}\nfor d in (glob.glob(\'/kaggle/input/*/volcache\') + glob.glob(\'/kaggle/input/*/*/volcache\') + glob.glob(\'/kaggle/input/*/*/*/volcache\')):\n    if os.path.isdir(d):\n        for f in os.listdir(d):\n            if f.endswith(\'.npz\'): MOUNTED[f[:-4]] = os.path.join(d, f)\nhits = _bglob(\'fused_labels_r1.csv\'); assert hits\nFL = pd.read_csv(hits[0]); FL[\'StudyInstanceUID\'] = FL[\'StudyInstanceUID\'].astype(str); FL = FL.set_index(\'StudyInstanceUID\')\nweak_ids = [s for s in ALL_IDS if s not in GOLD and s in FL.index]; gold_ids = sorted(GOLD)\nassert not [s for s in weak_ids + gold_ids if s not in MOUNTED]\ntlog(f\'studi={len(ALL_IDS)} gold={len(gold_ids)} cache={len(MOUNTED)} dev={DEVKIND}\')\nrep = train.set_index(\'StudyInstanceUID\')[\'Report\'] if \'Report\' in train.columns else None\ndef gkey(s):\n    r = None if rep is None else rep.get(s)\n    if r is None or isinstance(r, float) or not str(r).strip(): return \'u:\' + s\n    return \'r:\' + hashlib.md5(\' \'.join(str(r).split()).encode()).hexdigest()\ngroups = {}\nfor s in weak_ids: groups.setdefault(gkey(s), []).append(s)\nsizes = [0] * NFOLD; fold_of = {}\nfor gk, mem in sorted(groups.items(), key=lambda kv: (-len(kv[1]), kv[0])):\n    f = int(np.argmin(sizes))\n    for m in mem: fold_of[m] = f\n    sizes[f] += len(mem)\ntlog(f\'grup={len(groups)} lemah={len(weak_ids)} fold={sizes}\')\ndef label_arrays(ids):\n    y = FL.loc[ids, LAB].fillna(0.28).values.astype(np.float32)\n    c = np.stack([FL.loc[ids, f\'{t}__conf\'].fillna(0.15).values.astype(np.float32) for t in LAB], 1)\n    return y, (0.5 + np.clip(c, 0, 1)).astype(np.float32)\n\ndef centres(valid, k, train):\n    a, b = valid[0], valid[-1]\n    if train:\n        edges = np.linspace(a, b + 1, k + 1); c = edges[:-1] + np.random.rand(k) * (edges[1:] - edges[:-1])\n    else:\n        c = np.linspace(a, b, k)\n    c = np.round(c).astype(int)\n    return np.clip(c, a, b)\n\nclass SlotDS(torch.utils.data.Dataset):\n    """Satu item = satu studi: uint8 [NS, K, 3, H, W], mask [NS], pos [NS, K], target, bobot."""\n    def __init__(self, ids, Y=None, W=None, k=8, train=False): self.ids, self.Y, self.W, self.k, self.train = ids, Y, W, k, train\n    def __len__(self): return len(self.ids)\n    def __getitem__(self, i):\n        z = np.load(MOUNTED[self.ids[i]]); vol, m = z[\'vol\'], z[\'mask\']\n        H = vol.shape[-1]; x = np.zeros((NS, self.k, 3, H, H), np.uint8); mk = np.zeros(NS, bool); pos = np.zeros((NS, self.k), np.float32)\n        for s, (lo, hi) in enumerate(BOUNDS):\n            valid = [j for j in range(lo + 1, hi - 1) if m[j] > 0 and m[j - 1] > 0 and m[j + 1] > 0]\n            if len(valid) < 1: continue\n            c = centres(valid, self.k, self.train)\n            x[s] = np.stack([vol[[j - 1, j, j + 1]] for j in c]); mk[s] = True; pos[s] = (c - lo) / max(hi - lo - 1, 1)\n        y = self.Y[i] if self.Y is not None else np.zeros(N_T, np.float32)\n        w = self.W[i] if self.W is not None else np.ones(N_T, np.float32)\n        return torch.from_numpy(x), torch.from_numpy(mk), torch.from_numpy(pos), torch.from_numpy(np.asarray(y, np.float32)), torch.from_numpy(np.asarray(w, np.float32))\n\nMEAN_ = (0.485, 0.456, 0.406); STD_ = (0.229, 0.224, 0.225)\ndef make_views(x, res, train, group, fov=0.92):\n    """x uint8 [N,3,H,W] (N = n_series*group). Recorte FOV tengah + resize; train: rotasi/skala/geser/gamma/gain bersama per slot."""\n    n = x.shape[0]; ns = n // group; dev = x.device; x = x.float().div_(255.0)\n    if train:\n        ang = (torch.rand(ns, device=dev) - 0.5) * 2 * math.radians(12); sc = fov * (1 + (torch.rand(ns, device=dev) - 0.5) * 0.24)\n        tx = (torch.rand(ns, device=dev) - 0.5) * 0.12; ty = (torch.rand(ns, device=dev) - 0.5) * 0.12\n        gamma = torch.exp((torch.rand(ns, device=dev) - 0.5) * 0.5); gain = 1 + (torch.rand(ns, device=dev) - 0.5) * 0.2\n    else:\n        ang = torch.zeros(ns, device=dev); sc = torch.full((ns,), fov, device=dev); tx = ty = torch.zeros(ns, device=dev); gamma = gain = None\n    cos, sin = torch.cos(ang) * sc, torch.sin(ang) * sc\n    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1).repeat_interleave(group, 0)\n    grid = F.affine_grid(theta, (n, 3, res, res), align_corners=False)\n    x = F.grid_sample(x, grid, mode=\'bilinear\', padding_mode=\'zeros\', align_corners=False)\n    if train:\n        x = x.clamp_min(1e-4).pow(gamma.repeat_interleave(group).view(n, 1, 1, 1)) * gain.repeat_interleave(group).view(n, 1, 1, 1)\n    mean = torch.tensor(MEAN_, device=dev).view(1, 3, 1, 1); std = torch.tensor(STD_, device=dev).view(1, 3, 1, 1)\n    return (x - mean) / std\n\nclass KneeNet(nn.Module):\n    def __init__(self, backbone, dim=384, depth=2, drop=0.2, drop_path=0.1):\n        super().__init__()\n        import timm\n        self.enc = timm.create_model(backbone, pretrained=False, num_classes=0, drop_path_rate=drop_path)\n        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim), nn.LayerNorm(dim))\n        self.slot_emb = nn.Embedding(NS, dim)\n        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))\n        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)\n        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)\n        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, N_T))\n        self.cls_w = nn.Parameter(torch.randn(N_T, dim) * 0.02); self.cls_b = nn.Parameter(torch.zeros(N_T))\n    def forward(self, x, mask, pos, res, train=False):\n        b, s, k = x.shape[:3]\n        f = self.enc(make_views(x.reshape(b * s * k, *x.shape[3:]), res, train, k))      # semua slot dijalankan (bentuk tetap)\n        feat = f.view(b, s, k, -1)\n        slot = torch.arange(s, device=feat.device).view(1, s, 1).expand(b, s, k)\n        h = self.proj(feat) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1).to(feat.dtype))\n        h = h.view(b, s * k, -1)\n        pad = ~mask.unsqueeze(-1).expand(b, s, k).reshape(b, s * k)\n        pad = pad & ~pad.all(1, keepdim=True)\n        h = self.tf(h, src_key_padding_mask=pad)\n        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)\n        pooled = torch.einsum(\'btc,btd->bcd\', a.to(h.dtype), h)\n        return (pooled * self.cls_w).sum(-1) + self.cls_b\n\ndef make_model():\n    m = KneeNet(\'convnext_tiny\')\n    wf = sorted({os.path.realpath(p) for p in _bglob(\'model.safetensors\') if \'convnext-tiny-in22k\' in p})\n    assert len(wf) == 1, wf\n    from safetensors.torch import load_file\n    r = m.enc.load_state_dict(load_file(wf[0]), strict=False)\n    tlog(f\'bobot convnext_tiny IN22k: missing={list(r.missing_keys)[:4]} unexpected={list(r.unexpected_keys)[:4]}\')\n    assert len(r.missing_keys) == 0\n    return m\n\ndef auc1(yy, ss):\n    pos, neg = ss[yy == 1], ss[yy == 0]\n    if len(pos) == 0 or len(neg) == 0: return np.nan\n    r = pd.Series(np.concatenate([pos, neg])).rank().values\n    return float((r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))\nYg = train.set_index(\'StudyInstanceUID\').loc[gold_ids, LAB].values.astype(float)\ndef macro(P, Y): return float(np.nanmean([auc1(Y[:, j], P[:, j]) for j in range(N_T)]))\n\n@torch.no_grad()\ndef predict(model_, ids):\n    model_.eval(); out = []\n    dl = torch.utils.data.DataLoader(SlotDS(ids, k=K_EVAL), batch_size=BS, shuffle=False, num_workers=NW)\n    for x, mk, pos, _, _ in dl:\n        n = x.shape[0]\n        if n < BS:      # bentuk tetap utk XLA: isi ulang batch terakhir\n            pad = BS - n; x = torch.cat([x, x[:1].expand(pad, *x.shape[1:])]); mk = torch.cat([mk, mk[:1].expand(pad, *mk.shape[1:])]); pos = torch.cat([pos, pos[:1].expand(pad, *pos.shape[1:])])\n        with AC():\n            p = torch.sigmoid(model_(x.to(DEV), mk.to(DEV), pos.to(DEV), RES).float())\n        mark(); out.append(p[:n].cpu().numpy())\n    return np.concatenate(out)\n\nclass EMA:\n    def __init__(self, model, decay=0.998):\n        self.decay = decay; self.n = 0; self.m = copy.deepcopy(model).eval()\n        for p in self.m.parameters(): p.requires_grad_(False)\n    @torch.no_grad()\n    def update(self, model):\n        self.n += 1; d = self.decay if DEVKIND == \'tpu\' else min(self.decay, (1 + self.n) / (10 + self.n))\n        ep, mp = list(self.m.parameters()), list(model.parameters())\n        torch._foreach_mul_(ep, d); torch._foreach_add_(ep, mp, alpha=1 - d)\n        for be, bm in zip(self.m.buffers(), model.buffers()): be.copy_(bm)\n\nfor FOLD in FOLDS:\n    np.random.seed(SEED * 100 + FOLD); random.seed(SEED * 100 + FOLD); torch.manual_seed(SEED * 100 + FOLD)\n    train_ids = [s for s in weak_ids if fold_of[s] != FOLD]; held_ids = [s for s in weak_ids if fold_of[s] == FOLD]\n    assert not ({gkey(s) for s in train_ids} & {gkey(s) for s in held_ids}) and not (set(train_ids) & GOLD)\n    Y, W = label_arrays(train_ids)\n    tlog(f\'=== FOLD {FOLD}/{NFOLD}: latih={len(train_ids)} held={len(held_ids)} gold={len(gold_ids)}\')\n    model = make_model().to(DEV)\n    pr = np.clip((W * Y).sum(0) / W.sum(0), 0.02, 0.98)\n    with torch.no_grad(): model.cls_b.copy_(torch.logit(torch.tensor(pr, dtype=torch.float32)).to(DEV))\n    enc_p = list(model.enc.parameters()); eid = {id(p) for p in enc_p}\n    head_p = [p for p in model.parameters() if id(p) not in eid]\n    opt = torch.optim.AdamW([{\'params\': enc_p, \'lr\': LR_ENC}, {\'params\': head_p, \'lr\': LR_HEAD}], weight_decay=WD)\n    base = [g[\'lr\'] for g in opt.param_groups]\n    spe = len(train_ids) // (BS * ACC); total = spe * EPOCHS; warm = spe\n    def set_lr(s):\n        sc = min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total)))\n        if DEVKIND == \'tpu\': sc = max(1, round(sc * 20)) / 20\n        for g, b in zip(opt.param_groups, base): g[\'lr\'] = b * sc\n    scaler = torch.amp.GradScaler(\'cuda\') if DEVKIND == \'gpu\' else None\n    ema = EMA(model, 0.998)\n    dl = torch.utils.data.DataLoader(SlotDS(train_ids, Y, W, k=KW, train=True), batch_size=BS, shuffle=True, drop_last=True,\n                                     num_workers=NW, prefetch_factor=2, persistent_workers=True)\n    tlog(f\'steps={total} ({spe}/epoch) bs={BS}x{ACC} res={RES} k={KW}x{NS} lr={LR_ENC}/{LR_HEAD}\')\n    step = 0\n    for ep in range(EPOCHS):\n        model.train(); t_ep = time.time(); eloss, nb = torch.zeros((), device=DEV), 0; opt.zero_grad(set_to_none=True)\n        for bi, (x, mk, pos, yb, wb) in enumerate(dl):\n            x, mk, pos, yb, wb = x.to(DEV), mk.to(DEV), pos.to(DEV), yb.to(DEV), wb.to(DEV)\n            with AC():\n                logits = model(x, mk, pos, RES, train=True)\n            loss = (F.binary_cross_entropy_with_logits(logits.float(), yb, reduction=\'none\') * wb).mean()\n            if scaler: scaler.scale(loss / ACC).backward()\n            else: (loss / ACC).backward()\n            eloss = eloss + loss.detach(); nb += 1\n            if (bi + 1) % ACC == 0:\n                set_lr(step)\n                if scaler: scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)\n                if scaler: scaler.step(opt); scaler.update()\n                else: opt.step()\n                opt.zero_grad(set_to_none=True); ema.update(model); step += 1\n            mark()\n            if (bi + 1) % 100 == 0: tlog(f\'f{FOLD} ep{ep+1} {(bi+1)*BS}/{len(train_ids)} loss~{float(eloss)/nb:.4f} {(time.time()-t_ep):.0f}s\')\n        Pe = predict(ema.m, gold_ids)\n        tlog(f\'=== f{FOLD} EPOCH {ep+1}/{EPOCHS} loss={float(eloss)/max(1,nb):.4f} ({time.time()-t_ep:.0f}s) | gold-58 diag ema={macro(Pe, Yg):.4f}\')\n    del dl\n    torch.save({\'arch\': \'kneenet_convnext_tiny\', \'res\': RES, \'k\': K_EVAL, \'model\': {k: v.detach().cpu() for k, v in ema.m.state_dict().items()}},\n               f\'/kaggle/working/{CFG["outtag"]}_f{FOLD}.pt\')\n    Ph = predict(ema.m, held_ids); Pg = predict(ema.m, gold_ids)\n    oof = pd.DataFrame(np.vstack([Ph, Pg]), columns=LAB); oof.insert(0, \'StudyInstanceUID\', held_ids + gold_ids)\n    oof[\'is_gold\'] = [0] * len(held_ids) + [1] * len(gold_ids); oof[\'fold\'] = [FOLD] * len(held_ids) + [-1] * len(gold_ids)\n    oof.to_csv(f\'/kaggle/working/oof_{CFG["outtag"]}_f{FOLD}.csv\', index=False)\n    Th = FL.loc[held_ids, LAB].fillna(0.28).values.astype(float)\n    cs = [np.corrcoef(Ph[:, j], Th[:, j])[0, 1] for j in range(N_T)]\n    tlog(f\'*** FOLD {FOLD} SELESAI | gold-58 (EMA) = {macro(Pg, Yg):.4f} | korelasi(OOF, label fusi) = {np.nanmean(cs):.4f}\')\n    del model, ema, opt\ntlog(\'WORKER DONE\')\n'
CFG = {'dev': 'tpu', 'res': 256, 'k': 8, 'epochs': 14, 'seed': 1, 'nfold': 6, 'bs': 2, 'acc': 4, 'k_eval': 12, 'outtag': 'kneenet_s1'}
FOLDS = [0]


In [ ]:

import os, sys, time, json, subprocess, glob
t0 = time.time()
def tlog(m): print(f'[main t={time.time()-t0:.0f}s] {m}', flush=True)
os.makedirs('/kaggle/working', exist_ok=True)
open('/kaggle/working/_ke_src.py', 'w').write(KE_SRC); open('/kaggle/working/worker.py', 'w').write(WORKER_SRC)
if CFG['dev'] == 'gpu':
    import torch; NG = max(1, torch.cuda.device_count())
else:
    NG = 1
assign = {g: [f for i, f in enumerate(FOLDS) if i % NG == g] for g in range(NG)}
tlog(f'dev={CFG["dev"]} NG={NG} assign={assign}')
procs = {}
for g, fl in assign.items():
    if not fl: continue
    env = dict(os.environ, FOLDS=','.join(map(str, fl)), CFG=json.dumps(CFG), NW=('8' if CFG['dev'] == 'tpu' else '3'), PYTHONUNBUFFERED='1')
    if CFG['dev'] == 'gpu': env['CUDA_VISIBLE_DEVICES'] = str(g)
    else: env['PJRT_DEVICE'] = 'TPU'
    fh = open(f'/kaggle/working/worker{g}.log', 'w')
    procs[g] = (subprocess.Popen([sys.executable, '-u', '/kaggle/working/worker.py'], env=env, stdout=fh, stderr=subprocess.STDOUT), fh)
pos = {g: 0 for g in procs}
def pump():
    for g in procs:
        with open(f'/kaggle/working/worker{g}.log') as fh:
            fh.seek(pos[g]); new = fh.read(); pos[g] = fh.tell()
        for line in new.splitlines(): print(f'[g{g}] {line}', flush=True)
while any(p.poll() is None for p, _ in procs.values()):
    time.sleep(10); pump()
pump()
for g, (p, fh) in procs.items(): tlog(f'pekerja g{g} rc={p.returncode}')
tlog('SELESAI')
